# Diabetic Retinopathy Severity Classification — Baseline Model

**Dataset:** APTOS 2019 Blindness Detection (Kaggle)  
**Task:** 5-class severity classification (0 = No DR → 4 = Proliferative DR)  
**Baseline Architecture:** Simple 3-block CNN trained from scratch (no pretrained weights)

---

## Setup: Point these paths to your local APTOS 2019 dataset

```
data/
├── train.csv          ← id_code, diagnosis columns
└── train_images/      ← retinal fundus images (.png or .jpeg)
```

Download from: https://www.kaggle.com/c/aptos2019-blindness-detection/data

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# ── CONFIG ─────────────────────────────────────────────────────────────────────
CSV_PATH   = 'data/train.csv'
IMG_DIR    = 'data/train_images'
IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_EPOCHS = 20
LR         = 1e-3
SEED       = 42
# ───────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 1. Data Exploration

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total samples: {len(df)}")
print(f"\nClass distribution:")
print(df['diagnosis'].value_counts().sort_index().to_string())

CLASS_NAMES = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
counts = df['diagnosis'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
axes[0].bar(CLASS_NAMES, counts.values, color=['#4CAF50','#FFC107','#FF9800','#F44336','#9C27B0'])
axes[0].set_title('Class Distribution (APTOS 2019)', fontsize=13)
axes[0].set_xlabel('DR Severity')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=CLASS_NAMES, autopct='%1.1f%%',
            colors=['#4CAF50','#FFC107','#FF9800','#F44336','#9C27B0'])
axes[1].set_title('Class Proportions', fontsize=13)

plt.tight_layout()
plt.savefig('results/class_distribution.png', bbox_inches='tight')
plt.show()
print("\nNote: Significant class imbalance — No DR and Moderate are overrepresented.")

In [ ]:
# Show sample images from each class
from PIL import Image

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle('Sample Retinal Fundus Images — One per DR Severity Class', fontsize=13)

for cls in range(5):
    sample = df[df['diagnosis'] == cls].iloc[0]['id_code']
    for ext in ['.png', '.jpeg', '.jpg']:
        path = os.path.join(IMG_DIR, sample + ext)
        if os.path.exists(path):
            img = Image.open(path).convert('RGB').resize((256, 256))
            break
    axes[cls].imshow(img)
    axes[cls].set_title(CLASS_NAMES[cls], fontsize=10)
    axes[cls].axis('off')

plt.tight_layout()
plt.savefig('results/sample_images.png', bbox_inches='tight')
plt.show()

## 2. Data Loading

In [ ]:
from dataset import load_data, APTOSDataset, get_transforms

os.makedirs('results', exist_ok=True)

train_loader, val_loader, test_loader, train_df, val_df, test_df = load_data(
    csv_path=CSV_PATH,
    img_dir=IMG_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_size=0.15,
    test_size=0.15,
    seed=SEED
)

## 3. Baseline Model — Simple 3-Block CNN

**Architecture choices that keep this intentionally simple:**
- No pretrained weights (trained from scratch)
- Only 3 conv blocks (limited depth)
- No class weighting or oversampling (class imbalance unaddressed)
- Minimal augmentation (only horizontal flip)
- Standard cross-entropy loss

In [ ]:
from model import BaselineCNN
import torch.nn as nn
import torch.optim as optim

model = BaselineCNN(num_classes=5, input_size=IMG_SIZE).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: BaselineCNN")
print(f"Trainable parameters: {total_params:,}")
print(f"\nArchitecture:\n{model}")

## 4. Training

In [ ]:
from train import train, evaluate, print_metrics

criterion = nn.CrossEntropyLoss()  # No class weights — baseline deliberately ignores imbalance
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    device=DEVICE,
    num_epochs=NUM_EPOCHS
)

torch.save(model.state_dict(), 'results/baseline_cnn.pth')
print("Model saved to results/baseline_cnn.pth")

## 5. Training Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, history['train_loss'], label='Train Loss', marker='o', ms=4)
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   marker='s', ms=4)
axes[0].set_title('Loss Curve — Baseline CNN')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['train_acc'], label='Train Acc', marker='o', ms=4)
axes[1].plot(epochs, history['val_acc'],   label='Val Acc',   marker='s', ms=4)
axes[1].set_title('Accuracy Curve — Baseline CNN')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/training_curves.png', bbox_inches='tight')
plt.show()

## 6. Evaluation on Test Set

In [ ]:
test_loss, test_acc, test_labels, test_preds = evaluate(model, test_loader, criterion, DEVICE)
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")
cm = print_metrics(test_labels, test_preds)

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 6))
short_names = ['No DR\n(0)', 'Mild\n(1)', 'Moderate\n(2)', 'Severe\n(3)', 'Proliferative\n(4)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names, ax=ax)
ax.set_title('Confusion Matrix — Baseline CNN (Test Set)', fontsize=13)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', bbox_inches='tight')
plt.show()

print("\nKey observation: Severe DR (class 3) and Mild (class 1) likely show poor recall")
print("due to class imbalance — this is a known limitation of this baseline.")

## 7. Grad-CAM Visualization

Grad-CAM shows which regions of the retinal image the CNN uses to make its prediction.
For a well-trained model, we expect activations over lesions like microaneurysms and hemorrhages.

In [ ]:
from gradcam import visualize_gradcam
from dataset import APTOSDataset, get_transforms

_, val_transform = get_transforms(IMG_SIZE)
test_dataset = APTOSDataset(test_df, IMG_DIR, transform=val_transform)

fig = visualize_gradcam(model, test_dataset, DEVICE, num_samples=8, seed=SEED)
fig.savefig('results/gradcam.png', bbox_inches='tight')
plt.show()

## 8. Summary & Baseline Limitations

| Metric | Baseline CNN |
|--------|-------------|
| Test Accuracy | see above |
| Macro F1 | see above |
| Severe DR Recall | likely low |

### Known limitations of this baseline (to be addressed in future iterations):

1. **Class imbalance unaddressed** — No DR (~49%) dominates; model likely biased toward majority classes
2. **No pretrained features** — Trained from scratch on only ~2,500 training images, far too few for a 5-class medical task
3. **Shallow architecture** — 3 conv blocks can't capture the fine-grained lesion features (microaneurysms, hemorrhages) needed for high-severity classes
4. **Minimal augmentation** — No color jitter, rotation, or scale variation; retinal images vary widely in brightness and FOV across clinical sites
5. **No dedicated handling of Severe DR** — This is the most clinically critical class (late-stage, vision-threatening) yet gets the fewest training examples

### Planned improvements:
- Transfer learning with ResNet/EfficientNet pretrained on ImageNet
- Class-weighted loss + minority oversampling
- Aggressive augmentation pipeline
- Attention mechanisms / Transformer backbone
- Ensemble or test-time augmentation